# Mixed reduced GP

continuous feature だけをPCA/PLSで圧縮し、categorical feature を native categorical covariance に残します。


## Mixedデータ


In [ ]:
import torch
from robotorchan.models import MixedPCAGP, MixedPLSGP

torch.manual_seed(0)
torch.set_default_dtype(torch.double)
continuous = torch.rand(18, 5)
category = torch.randint(0, 3, (18, 1)).double()
X = torch.cat([continuous, category], dim=-1)
Y = torch.sin(4 * continuous[:, :1]) + 0.2 * category


## Mixed reduction


In [ ]:
pca = MixedPCAGP(X, Y, n_components=2, cat_dims=[5])
pls = MixedPLSGP(X, Y, n_components=2, cat_dims=[5])
print(pca.raw_train_X.shape, pls.raw_train_X.shape)


## Original mixed-space posterior


In [ ]:
X_test = torch.cat([torch.rand(3, 5), torch.tensor([[0.0], [1.0], [2.0]])], dim=-1)
print(pca.posterior(X_test).mean.squeeze(-1))
print(pls.posterior(X_test).mean.squeeze(-1))


## GP fitting

PCA/PLSは連続変数のみを削減し、カテゴリ列は保持します。既存のMLLで両モデルを学習します。


In [ ]:
from botorch.fit import fit_gpytorch_mll

models = {"MixedPCA": pca, "MixedPLS": pls}
for name, model in models.items():
    fit_gpytorch_mll(model.make_mll())
    model.eval()
    print(name, "fitted")


## Mixed-space candidate proposal and evaluation

カテゴリ値0・1・2を明示的に列挙し、元の5連続変数と組み合わせた候補プールからposterior mean最大の点を選びます。


In [ ]:
def objective(candidate):
    return torch.sin(4 * candidate[..., :1]) + 0.2 * candidate[..., 5:6]


continuous_pool = torch.rand(64, 5)
candidate_pool = torch.cat(
    [
        torch.cat(
            [continuous_pool, torch.full((64, 1), float(category))],
            dim=-1,
        )
        for category in (0, 1, 2)
    ],
    dim=0,
)
for name, model in models.items():
    with torch.no_grad():
        scores = model.posterior(candidate_pool).mean.squeeze(-1)
    candidate = candidate_pool[scores.argmax()].unsqueeze(0)
    assert candidate.shape == torch.Size([1, 6])
    assert candidate[0, 5].item() in (0, 1, 2)
    print(name, "candidate:", candidate)
    print(name, "posterior mean:", scores.max())
    print(name, "objective:", objective(candidate))
